In [1]:
import cv2
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("dataset.csv")

In [3]:
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

In [4]:
def load_images(df, img_size=(64, 64)):
    X = []
    y = []
    for _, fila in df.iterrows():
        img_bgr = cv2.imread(fila['filepath'])
        img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        img = cv2.resize(img_gray, img_size)
        img = img / 255.0
        X.append(img)
        y.append(fila['label'])
    X = np.array(X).reshape(-1, img_size[0], img_size[1], 1)
    y = np.array(y)
    return X, y

In [5]:
X_train, y_train = load_images(train_df)
X_test, y_test = load_images(val_df)
print(X_train.shape)
print(X_test.shape)

(38400, 64, 64, 1)
(9600, 64, 64, 1)


In [6]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(64, 64, 1)),
    
    layers.Conv2D(16, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

In [7]:
from tensorflow.keras.callbacks import EarlyStopping

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [8]:
epocas = 20
lote = 32

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=epocas,
    batch_size=lote,
    callbacks=[early_stopping]
)

Epoch 1/20
   1/1200 ━━━━━━━━━━━━━━━━━━━━ 6:54 346ms/step - accuracy: 0.5938 - loss: 0.6865

error: operand #1 does not dominate this use
E0000 00:00:1789421975.568204  150219 meta_optimizer.cc:967] tfg_optimizer{any(tfg-consolidate-attrs,tfg-toposort,tfg-shape-inference{graph-version=0},tfg-prepare-attrs-export)} failed: INVALID_ARGUMENT: MLIR Graph Optimizer failed: 
W0000 00:00:1789421975.568460  150219 optimize_function_graph_utils.cc:634] Ignoring multi-device function optimization failure: INVALID_ARGUMENT: MLIR Graph Optimizer failed: 
error: operand #0 does not dominate this use
W0000 00:00:1789421975.664607  150219 optimize_function_graph_utils.cc:634] Ignoring multi-device function optimization failure: INVALID_ARGUMENT: MLIR Graph Optimizer failed: 


1198/1200 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8433 - loss: 0.3080

error: operand #0 does not dominate this use
W0000 00:00:1789421994.872130  150219 optimize_function_graph_utils.cc:634] Ignoring multi-device function optimization failure: INVALID_ARGUMENT: MLIR Graph Optimizer failed: 


1200/1200 ━━━━━━━━━━━━━━━━━━━━ 21s 17ms/step - accuracy: 0.9186 - loss: 0.1914 - val_accuracy: 0.9711 - val_loss: 0.0864
Epoch 2/20
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 24s 20ms/step - accuracy: 0.9687 - loss: 0.0908 - val_accuracy: 0.9766 - val_loss: 0.0647
Epoch 3/20
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 26s 22ms/step - accuracy: 0.9749 - loss: 0.0721 - val_accuracy: 0.9744 - val_loss: 0.0642
Epoch 4/20
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 22s 19ms/step - accuracy: 0.9812 - loss: 0.0544 - val_accuracy: 0.9786 - val_loss: 0.0572
Epoch 5/20
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 24s 20ms/step - accuracy: 0.9847 - loss: 0.0441 - val_accuracy: 0.9877 - val_loss: 0.0378
Epoch 6/20
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 23s 19ms/step - accuracy: 0.9874 - loss: 0.0376 - val_accuracy: 0.9885 - val_loss: 0.0372
Epoch 7/20
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 24s 20ms/step - accuracy: 0.9893 - loss: 0.0331 - val_accuracy: 0.9894 - val_loss: 0.0305
Epoch 8/20
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 26s 22ms/step - accuracy: 0.9907 - loss: 0.02

In [ ]:
from pathlib import Path

model_path = Path("..") / "models" / "model_eye.keras"
model.save(model_path)